In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l4.2 Pre-normalization

PocketLM normalizes *before* each sublayer (`attn(norm(x))`), not after. Pre-norm
keeps the residual stream clean and is what lets modern transformers train deep
without careful warmup tricks.

In [ ]:
from torch import nn

torch.manual_seed(0)
dim = 8
ln = nn.LayerNorm(dim)
x = torch.randn(1, 4, dim) * 5 + 3   # arbitrary scale and shift
normed = ln(x)
row = normed[0, 0]
print("each position is normalized: mean", round(row.mean().item(), 6),
      "std", round(row.std(unbiased=False).item(), 4))

LayerNorm rescales each position vector to mean 0, unit variance (then a learned
affine). That stable input is what each sublayer sees.

In [ ]:
assert abs(row.mean().item()) < 1e-5
assert abs(row.std(unbiased=False).item() - 1.0) < 1e-4